In [1]:
import numpy as np
from tqdm import tqdm

The neural network we are going to recreate is made up of only one neuron. Basically, it is just a perceptron and we are going to try to optimize that.

An Important insight is that these neural networks cannot really generate negative values here. It is a leaky abstraction. Because of the fact that we are using relu activation we are unable to generate a negative value. Because it just clips it down to a negative value. And, also the gradient is such that if the activations are negative. Then, the gradient should be zero.

In [1054]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4], [5, 1, 5], [-1, 3, 6], [-2, 10, 7], [1, -4, 8], [2, 5, 9], [0, 3, 10], [1, 0, 11], [-2, 20, 12]])
Y = np.array([11, 13, 19, 18, 21, -2, 0, -3, 9, -2, -1, 5])

In [1055]:
weight = np.array([1, 1, 1])
bias = np.array(1)

In [1056]:
N = len(X)
learning_rate = 0.001

In [1060]:
for i in tqdm(range(100000)):
    activations = np.maximum(0, X @ weight + bias)
    error = activations - Y

    error[activations <= 0] = 0


    gradient_w = 2/N *  np.sum( error * X.T , axis=1)
    gradient_b = 2/N *  np.sum(error)





    weight = weight - learning_rate*gradient_w
    bias = bias - learning_rate*gradient_b

    # if i % 100 == 0:
    # print(f"Weight = {weight} Bias = {bias}")

100%|██████████| 100000/100000 [00:02<00:00, 36883.38it/s]


In [1061]:
print(f"Weight = {weight} Bias = {bias}")

Weight = [ 4.00000012  1.00000004 -1.00000001] Bias = 4.9999995530323265


Another important insight is that if it jumps too far in the negative, it is hard for it to recover because the gradient then just collapses immmediately.

In [130]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4]])
Y = np.array([[123], [129], [243], [168]])


In [6]:
# W1 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
# B1 = np.array([1, 1, 0, 1])

# W2 = np.array([[1, 2, 3, 4]])
# B2 = np.array([2])
# W1.shape, W2.shape, B1.shape, B2.shape

((4, 3), (1, 4), (4,), (1,))

In [18]:
# h1 = W1 @ X[3] + B1
# W2 @ h1 + B2

array([168])

In [196]:
X = X.reshape(4, 3, 1)
Y = Y.reshape(4, 1, 1)

In [298]:
np.random.randint(1, 5, size=(4, 3))

array([[4, 4, 1],
       [3, 4, 2],
       [4, 4, 1],
       [4, 4, 3]])

In [301]:
class NeuralNetwork():
    def __init__(self, X, Y):

        # np.random.randint(1, 5, size=(4, 3))

        W0 = np.random.randint(1, 5, size=(4, 3))
        B0 = np.random.randint(1, 5, size=(4, 1))
        W1 = np.random.randint(1, 5, size=(1, 4))
        B1 = np.random.randint(1, 5, size=(1, 1))

        self.W = [W0, W1]
        self.B = [B0, B1]

        self.X = X
        self.Y = Y

        self.f = []
        self.h = []

        self.gradient_f = []
        self.gradient_W = []
        self.gradient_B = []

    def forward_pass(self, x):

        f0 = self.W[0] @ x + self.B[0]
        h1 = np.maximum(f0, 0)
        f1 = self.W[1] @ h1 + self.B[1]

        self.f = [f0, f1]
        self.h = [0, h1]

    def get_gradients(self, x, y):
        diagonal_matrix = np.diag(self.f[0].flatten())
        np.fill_diagonal(diagonal_matrix, np.maximum(diagonal_matrix.diagonal(), 0))

        gradient_f1 = -2 * (y - self.f[1])
        gradient_f0 = gradient_f1 @ self.W[1] @ diagonal_matrix

        gradient_B0 = gradient_f0.T
        gradient_B1 = gradient_f1.T

        gradient_W0 = (x @ gradient_f0).T
        gradient_W1 = (self.h[1] @ gradient_f1).T

        self.gradient_f.append( [gradient_f0, gradient_f1] )
        self.gradient_W.append( [gradient_W0, gradient_W1] )
        self.gradient_B.append( [gradient_B0, gradient_B1] )

    def gradient_descent(self, learning_rate= 0.0001):

        gradient_B0 = np.mean([sublist[0] for sublist in self.gradient_B])
        gradient_B1 = np.mean([sublist[1] for sublist in self.gradient_B])
        gradient_W0 = np.mean([sublist[0] for sublist in self.gradient_W])
        gradient_W1 = np.mean([sublist[1] for sublist in self.gradient_W])

        self.B[0] = self.B[0] - learning_rate*gradient_B0
        self.B[1] = self.B[1] - learning_rate*gradient_B1
        self.W[0] = self.W[0] - learning_rate*gradient_W0
        self.W[1] = self.W[1] - learning_rate*gradient_W1

    def train(self, epochs = 100):
        for _ in tqdm(range(epochs)):

            self.gradient_f = []
            self.gradient_W = []
            self.gradient_B = []

            for i in range(len(X)):

                self.forward_pass(X[i])
                self.get_gradients(X[i], Y[i])
                self.gradient_descent()

In [302]:
nn = NeuralNetwork(X, Y)


In [316]:
nn.train(epochs=10000)

100%|██████████| 10000/10000 [00:15<00:00, 637.22it/s]


In [311]:
Y

array([[[123]],

       [[129]],

       [[243]],

       [[168]]])

In [317]:
for i in range(4):
    nn.forward_pass(X[i])
    print(nn.f[1])

[[118.90751244]]
[[130.14707276]]
[[232.37061527]]
[[175.37218895]]


In [308]:
nn.W, nn.B

([array([[ 1.92888732,  1.92888732,  1.92888732],
         [-0.07111268,  0.92888732,  0.92888732],
         [ 0.92888732,  1.92888732,  1.92888732],
         [ 1.92888732,  1.92888732,  0.92888732]]),
  array([[3.75835246, 4.75835246, 1.75835246, 3.75835246]])],
 [array([[0.69311864],
         [1.69311864],
         [0.69311864],
         [0.69311864]]),
  array([[2.79271922]])])

In [189]:
nn.train(epochs=1)

IndexError: list index out of range

In [166]:
def forward_pass(x, W, B):

    f0 = W[0] @ x + B[0]
    h1 = np.maximum(f0, 0)
    f1 = W[1] @ h1 + B[1]

    return f0, h1, f1


In [172]:
def get_gradients(x, y, f0, h1, f1, W, B):
    diagonal_matrix = np.diag(f0.flatten())
    np.fill_diagonal(diagonal_matrix, np.maximum(diagonal_matrix.diagonal(), 0))

    gradient_f1 = -2 * (y - f1)
    gradient_f0 = gradient_f1 @ W[1] @ diagonal_matrix

    gradient_B0 = gradient_f0.T
    gradient_B1 = gradient_f1.T

    gradient_W0 = (x @ gradient_f0).T
    gradient_W1 = (h1 @ gradient_f1).T


    return gradient_W0, gradient_B0, gradient_W1, gradient_B1

In [173]:
f0, h1, f1 = forward_pass(X[0], W, B)

In [174]:
gradient_W0, gradient_B0, gradient_W1, gradient_B1 = get_gradients(X[0], Y[0], f0, h1, f1, W, B)

In [179]:
def gradient_descent(W, B, learning_rate= 0.0001):

    W[0] = W[0] - learning_rate*gradient_W0
    B[0] = B[0] - learning_rate*gradient_B0
    W[1] = W[1] - learning_rate*gradient_W1
    B[1] = B[1] - learning_rate*gradient_B1

    return W, B

In [176]:
W, B = gradient_descent(W, B)

In [177]:
W, B

([array([[1.1176, 1.3528, 1.1176],
         [1.1176, 1.3528, 1.1176],
         [1.1176, 1.3528, 1.1176],
         [1.1176, 1.3528, 1.1176]]),
  array([[1.1176, 1.1176, 1.1176, 1.1176]])],
 [array([[1.1176],
         [1.1176],
         [1.1176],
         [1.1176]]),
  array([[1.0196]])])